# 📰 Fake News Detection — Exploratory Data Analysis

This notebook explores the [Kaggle Fake News Dataset](https://www.kaggle.com/c/fake-news/data).

**Dataset columns:**
- `id` — unique article identifier
- `title` — article headline
- `author` — article author
- `text` — full article body
- `label` — `1` = Fake News, `0` = Real News

## 1. Setup

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src/ to path so we can reuse utility functions
sys.path.insert(0, os.path.join("..", "src"))

# Visualisation settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

DATA_PATH = os.path.join("..", "data", "train.csv")

## 2. Load & Inspect the Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

## 3. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df[missing_df["Missing Count"] > 0]

In [ ]:
# Fill missing values (same as preprocessing pipeline)
for col in ("author", "title", "text"):
    df[col] = df[col].fillna("")

print("Missing values after filling:", df.isnull().sum().sum())

## 4. Label Distribution

In [ ]:
label_counts = df["label"].value_counts()
label_names = {0: "Real", 1: "Fake"}

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# Count plot
sns.countplot(
    data=df, x="label", palette=["steelblue", "tomato"],
    order=[0, 1], ax=ax[0]
)
ax[0].set_xticklabels(["Real (0)", "Fake (1)"])
ax[0].set_title("Article Count by Label")
ax[0].set_xlabel("Label")
ax[0].set_ylabel("Count")

# Pie chart
ax[1].pie(
    label_counts.values,
    labels=[label_names[l] for l in label_counts.index],
    autopct="%1.1f%%",
    colors=["steelblue", "tomato"],
    startangle=90,
)
ax[1].set_title("Label Distribution")

plt.tight_layout()
plt.show()

## 5. Text Length Distribution

In [ ]:
df["text_length"] = df["text"].apply(len)
df["title_length"] = df["title"].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(
    axes,
    ["text_length", "title_length"],
    ["Article Text Length", "Title Length"],
):
    for label, color, name in [(0, "steelblue", "Real"), (1, "tomato", "Fake")]:
        subset = df[df["label"] == label][col]
        ax.hist(subset.clip(upper=subset.quantile(0.99)), bins=50, alpha=0.6, color=color, label=name)
    ax.set_title(title)
    ax.set_xlabel("Character Count")
    ax.set_ylabel("Frequency")
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Word Cloud — Fake vs Real News

In [ ]:
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print("wordcloud not installed — skipping. Install with: pip install wordcloud")

if HAS_WORDCLOUD:
    from utils import clean_text

    fake_text = " ".join(df[df["label"] == 1]["text"].astype(str).apply(clean_text).tolist())
    real_text = " ".join(df[df["label"] == 0]["text"].astype(str).apply(clean_text).tolist())

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, text, title, cmap in [
        (axes[0], fake_text, "Fake News — Top Words", "Reds"),
        (axes[1], real_text, "Real News — Top Words", "Blues"),
    ]:
        wc = WordCloud(
            width=800, height=400,
            background_color="white",
            colormap=cmap,
            max_words=100,
        ).generate(text)
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(title, fontsize=16)

    plt.tight_layout()
    plt.show()

## 7. Most Common Words

In [ ]:
from collections import Counter
from utils import clean_text

def top_n_words(series, n=20):
    """Return the top-n most common tokens in a text Series."""
    all_tokens = []
    for text in series:
        all_tokens.extend(clean_text(text).split())
    return Counter(all_tokens).most_common(n)

top_fake = top_n_words(df[df["label"] == 1]["text"])
top_real = top_n_words(df[df["label"] == 0]["text"])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, top_words, title, color in [
    (axes[0], top_fake, "Top 20 Words in Fake News", "tomato"),
    (axes[1], top_real, "Top 20 Words in Real News", "steelblue"),
]:
    words, counts = zip(*top_words)
    sns.barplot(x=list(counts), y=list(words), color=color, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Frequency")
    ax.set_ylabel("Word")

plt.tight_layout()
plt.show()

## 8. Key Takeaways

- The dataset is roughly balanced between fake and real articles.
- Fake articles tend to be longer on average.
- Common political and media terms dominate both classes.
- After preprocessing (stopword removal, stemming), TF-IDF features effectively separate the two classes.

Proceed to model training with `src/train.py`.